# FreshCheck 50-Seed Benchmark Runner

This notebook runs repeated-seed experiments for FreshCheck on Colab.

Default mode is safe dry-run. Set `RUN_EXPERIMENTS = True` only after cells 1-8 show the correct commands and paths.

Experiments covered:

- A: source/Kaggle only -> target holdout
- B: source + target adaptation -> target holdout
- C: balanced mixed source + target -> target holdout
- D: source-retention check using Method C checkpoints -> source holdout
- C+: Method C with hyperparameter tuning
- C+Patch: C+ with random patch sampling
- Structured Multi-View: deterministic multi-view patch sampling


In [ ]:
#@title 1) Mount Drive and clone/update repository
from google.colab import drive
from pathlib import Path
import os, subprocess, json, time, shutil, glob

drive.mount('/content/drive')

REPO_URL = 'https://github.com/techasit239/DADS7202_PigPicture.git'
REPO_DIR = Path('/content/DADS7202_PigPicture')

if not REPO_DIR.exists():
    subprocess.run(f'git clone {REPO_URL} {REPO_DIR}', shell=True, check=True)
else:
    subprocess.run('git fetch origin main', shell=True, cwd=REPO_DIR, check=True)
    subprocess.run('git checkout main', shell=True, cwd=REPO_DIR, check=True)
    subprocess.run('git pull origin main', shell=True, cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print('repo:', REPO_DIR)
print('cwd:', os.getcwd())


In [ ]:
#@title 2) Install dependencies
import subprocess, sys, os
subprocess.run('python -m pip install -q --upgrade pip', shell=True, check=True)
subprocess.run('python -m pip install -q -r requirements.txt', shell=True, check=True)


In [ ]:
#@title 3) Benchmark configuration
from pathlib import Path

# Safety switch. Keep False for dry-run command preview.
RUN_EXPERIMENTS = False  #@param {type:'boolean'}
RUN_MODE = 'pilot_10'    #@param ['pilot_10', 'full_50', 'custom']
CUSTOM_SEEDS = '42,52,62,72,82'  # used only when RUN_MODE='custom'

RUN_EXP_A = True          #@param {type:'boolean'}
RUN_EXP_B = True          #@param {type:'boolean'}
RUN_EXP_C = True          #@param {type:'boolean'}
RUN_EXP_D = True          #@param {type:'boolean'}
RUN_EXP_CPLUS = True      #@param {type:'boolean'}
RUN_EXP_CPLUS_PATCH = True #@param {type:'boolean'}
RUN_STRUCTURED = True     #@param {type:'boolean'}

# Choose any subset, e.g. 'efficientnet_b0 swin_t'
MODELS = 'efficientnet_b0 swin_t convnext_tiny dinov2_vits14'  #@param {type:'string'}

if RUN_MODE == 'pilot_10':
    SEEDS = [42 + 10*i for i in range(10)]
elif RUN_MODE == 'full_50':
    SEEDS = [42 + 10*i for i in range(50)]
else:
    SEEDS = [int(x.strip()) for x in CUSTOM_SEEDS.split(',') if x.strip()]

MODEL_LIST = MODELS.split()
DRIVE_ROOT = Path('/content/drive/MyDrive/FreshCheck')
ARTIFACTS_ROOT = DRIVE_ROOT / 'artifacts'
BENCH_ROOT = ARTIFACTS_ROOT / 'benchmark_50seed'
SUMMARY_ROOT = BENCH_ROOT / '_summary'
LOG_ROOT = BENCH_ROOT / '_logs'
for p in [BENCH_ROOT, SUMMARY_ROOT, LOG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('RUN_EXPERIMENTS:', RUN_EXPERIMENTS)
print('run mode:', RUN_MODE)
print('n seeds:', len(SEEDS), SEEDS[:10], '...' if len(SEEDS) > 10 else '')
print('models:', MODEL_LIST)
print('output root:', BENCH_ROOT)


In [ ]:
#@title 4) Dataset and manifest paths
from pathlib import Path

EXPERIMENTS_AUTO = ARTIFACTS_ROOT / 'experiments_auto'
EXPERIMENTS_MANUAL = ARTIFACTS_ROOT / 'experiments_manual'
TARGET_SPLIT = ARTIFACTS_ROOT / 'target_split'
MANIFESTS = ARTIFACTS_ROOT / 'manifests'

A_TRAIN = EXPERIMENTS_AUTO / 'exp_a_train.csv'
A_VAL = EXPERIMENTS_AUTO / 'exp_a_val.csv'
B_TRAIN = EXPERIMENTS_AUTO / 'exp_b_train.csv'
B_VAL = EXPERIMENTS_AUTO / 'exp_b_val.csv'

C_TRAIN = EXPERIMENTS_MANUAL / 'exp_c_train.csv'
C_VAL = EXPERIMENTS_MANUAL / 'exp_c_val.csv'
if not C_TRAIN.exists():
    C_TRAIN = EXPERIMENTS_AUTO / 'exp_c_train_rebalanced.csv'
if not C_VAL.exists():
    C_VAL = EXPERIMENTS_AUTO / 'exp_c_val.csv'

TARGET_HOLDOUT = TARGET_SPLIT / 'target_holdout.csv'
if not TARGET_HOLDOUT.exists():
    TARGET_HOLDOUT = EXPERIMENTS_AUTO / 'target_holdout.csv'

SOURCE_HOLDOUT = EXPERIMENTS_AUTO / 'source_holdout.csv'
if not SOURCE_HOLDOUT.exists():
    SOURCE_HOLDOUT = MANIFESTS / 'source_holdout_manifest.csv'

PATHS = {
    'A_TRAIN': A_TRAIN, 'A_VAL': A_VAL,
    'B_TRAIN': B_TRAIN, 'B_VAL': B_VAL,
    'C_TRAIN': C_TRAIN, 'C_VAL': C_VAL,
    'TARGET_HOLDOUT': TARGET_HOLDOUT,
    'SOURCE_HOLDOUT': SOURCE_HOLDOUT,
}
for name, path in PATHS.items():
    print(f'{name:16s}', 'OK' if path.exists() else 'MISSING', path)


In [ ]:
#@title 5) Training profiles
COMMON_FAST = dict(
    epochs=10, patience=3, lr=1e-4, weight_decay=1e-2, dropout=0.30,
    img_size=224, batch_size=16, num_workers=4, prefetch_factor=4,
    freeze_backbone_epochs=0, sampling_mode='single', num_patches=1,
)

CPLUS_PROFILE = dict(
    epochs=22, patience=6, lr=3e-4, weight_decay=5e-3, dropout=0.25,
    img_size=224, batch_size=32, num_workers=4, prefetch_factor=4,
    freeze_backbone_epochs=2, sampling_mode='single', num_patches=1,
)

PATCH_PROFILE = dict(
    epochs=18, patience=5, lr=3e-4, weight_decay=5e-3, dropout=0.25,
    img_size=224, batch_size=32, num_workers=4, prefetch_factor=4,
    freeze_backbone_epochs=2, sampling_mode='patch', num_patches=4,
)

STRUCTURED_PROFILE = dict(
    epochs=14, patience=5, lr=2e-4, weight_decay=1e-2, dropout=0.30,
    img_size=224, batch_size=16, num_workers=4, prefetch_factor=4,
    freeze_backbone_epochs=2, sampling_mode='structured', num_patches=3,
)

MODEL_OVERRIDES = {
    'efficientnet_b0': dict(batch_size=32),
    'swin_t': dict(batch_size=16),
    'convnext_tiny': dict(batch_size=24),
    'dinov2_vits14': dict(batch_size=16),
}

def profile_for(model, base):
    p = dict(base)
    p.update(MODEL_OVERRIDES.get(model, {}))
    return p


In [ ]:
#@title 6) Command helpers
import subprocess, shlex, json, os, time
from pathlib import Path

def run_cmd(cmd, log_path=None, dry_run=False):
    print('\n$ ' + cmd)
    if dry_run:
        return 0
    started = time.time()
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    elapsed = time.time() - started
    text = ''
    if result.stdout:
        text += result.stdout
        print(result.stdout)
    if result.stderr:
        text += '\n[stderr]\n' + result.stderr
        print(result.stderr)
    text += f'\n[exit_code] {result.returncode}\n[elapsed_sec] {elapsed:.1f}\n'
    if log_path:
        Path(log_path).parent.mkdir(parents=True, exist_ok=True)
        Path(log_path).write_text(text, encoding='utf-8')
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')
    return result.returncode

def model_checkpoint(train_dir, model):
    return Path(train_dir) / 'checkpoints' / f'{model}_best.pt'

def eval_json(eval_dir, model):
    return Path(eval_dir) / f'{model}_eval.json'

def build_train_cmd(train_csv, val_csv, out_dir, model, seed, profile):
    p = profile
    flags = []
    if p.get('sampling_mode') == 'patch':
        flags.append('--patch-sampling')
    if p.get('sampling_mode') in ('patch', 'structured'):
        flags += ['--sampling-mode', str(p['sampling_mode']), '--num-patches', str(p['num_patches'])]
    if p.get('freeze_backbone_epochs', 0) > 0:
        flags += ['--freeze-backbone-epochs', str(p['freeze_backbone_epochs'])]
    flags += ['--amp', '--amp-dtype', 'bf16', '--allow-tf32', '--persistent-workers']
    return f'''python /content/DADS7202_PigPicture/run_freshcheck.py train \
  --train-csv "{train_csv}" \
  --val-csv "{val_csv}" \
  --output-dir "{out_dir}" \
  --models {model} \
  --epochs {p['epochs']} \
  --patience {p['patience']} \
  --lr {p['lr']} \
  --weight-decay {p['weight_decay']} \
  --dropout {p['dropout']} \
  --img-size {p['img_size']} \
  --batch-size {p['batch_size']} \
  --num-workers {p['num_workers']} \
  --prefetch-factor {p['prefetch_factor']} \
  {' '.join(flags)} \
  --seed {seed}'''

def build_eval_cmd(csv_path, out_dir, model, checkpoint, profile):
    p = profile
    flags = []
    if p.get('sampling_mode') == 'patch':
        flags.append('--patch-sampling')
    if p.get('sampling_mode') in ('patch', 'structured'):
        flags += ['--sampling-mode', str(p['sampling_mode']), '--num-patches', str(p['num_patches'])]
    return f'''python /content/DADS7202_PigPicture/run_freshcheck.py evaluate \
  --csv "{csv_path}" \
  --output-dir "{out_dir}" \
  --models {model} \
  --checkpoint-paths {model}="{checkpoint}" \
  --batch-size {p['batch_size']} \
  --num-workers {p['num_workers']} \
  {' '.join(flags)}'''

def run_train_eval(exp_name, model, seed, train_csv, val_csv, target_csv, source_csv, profile, dry_run=True):
    root = BENCH_ROOT / exp_name / model / f'seed_{seed}'
    train_dir = root / 'train'
    eval_target_dir = root / 'eval_target'
    eval_source_dir = root / 'eval_source'
    ckpt = model_checkpoint(train_dir, model)
    if not ckpt.exists():
        log = LOG_ROOT / exp_name / model / f'seed_{seed}_train.log'
        run_cmd(build_train_cmd(train_csv, val_csv, train_dir, model, seed, profile), log, dry_run=dry_run)
    else:
        print('skip existing checkpoint:', ckpt)
    if ckpt.exists() or dry_run:
        if not eval_json(eval_target_dir, model).exists():
            log = LOG_ROOT / exp_name / model / f'seed_{seed}_eval_target.log'
            run_cmd(build_eval_cmd(target_csv, eval_target_dir, model, ckpt, profile), log, dry_run=dry_run)
        if source_csv and Path(source_csv).exists() and not eval_json(eval_source_dir, model).exists():
            log = LOG_ROOT / exp_name / model / f'seed_{seed}_eval_source.log'
            run_cmd(build_eval_cmd(source_csv, eval_source_dir, model, ckpt, profile), log, dry_run=dry_run)

def run_eval_only(exp_name, model, seed, csv_path, source_exp_name, profile, eval_subdir='eval_source', dry_run=True):
    root = BENCH_ROOT / exp_name / model / f'seed_{seed}'
    out_dir = root / eval_subdir
    source_ckpt = model_checkpoint(BENCH_ROOT / source_exp_name / model / f'seed_{seed}' / 'train', model)
    if not source_ckpt.exists() and not dry_run:
        print('skip D missing source checkpoint:', source_ckpt)
        return
    if not eval_json(out_dir, model).exists():
        log = LOG_ROOT / exp_name / model / f'seed_{seed}_{eval_subdir}.log'
        run_cmd(build_eval_cmd(csv_path, out_dir, model, source_ckpt, profile), log, dry_run=dry_run)


In [ ]:
#@title 7) Define experiment matrix
EXPERIMENTS = []
if RUN_EXP_A:
    EXPERIMENTS.append(dict(name='exp_a', kind='train_eval', train=A_TRAIN, val=A_VAL, target=TARGET_HOLDOUT, source=SOURCE_HOLDOUT, profile=COMMON_FAST))
if RUN_EXP_B:
    EXPERIMENTS.append(dict(name='exp_b', kind='train_eval', train=B_TRAIN, val=B_VAL, target=TARGET_HOLDOUT, source=SOURCE_HOLDOUT, profile=COMMON_FAST))
if RUN_EXP_C:
    EXPERIMENTS.append(dict(name='exp_c', kind='train_eval', train=C_TRAIN, val=C_VAL, target=TARGET_HOLDOUT, source=SOURCE_HOLDOUT, profile=COMMON_FAST))
if RUN_EXP_D:
    EXPERIMENTS.append(dict(name='exp_d_source_retention_from_c', kind='eval_only', csv=SOURCE_HOLDOUT, checkpoint_experiment='exp_c', profile=COMMON_FAST))
if RUN_EXP_CPLUS:
    EXPERIMENTS.append(dict(name='exp_cplus', kind='train_eval', train=C_TRAIN, val=C_VAL, target=TARGET_HOLDOUT, source=SOURCE_HOLDOUT, profile=CPLUS_PROFILE))
if RUN_EXP_CPLUS_PATCH:
    EXPERIMENTS.append(dict(name='exp_cplus_patch', kind='train_eval', train=C_TRAIN, val=C_VAL, target=TARGET_HOLDOUT, source=SOURCE_HOLDOUT, profile=PATCH_PROFILE))
if RUN_STRUCTURED:
    EXPERIMENTS.append(dict(name='exp_structured_multiview', kind='train_eval', train=C_TRAIN, val=C_VAL, target=TARGET_HOLDOUT, source=SOURCE_HOLDOUT, profile=STRUCTURED_PROFILE))

print('experiments:')
for e in EXPERIMENTS:
    print('-', e['name'], e['kind'])


In [ ]:
#@title 8) Dry-run preview first two commands
preview_models = MODEL_LIST[:1]
preview_seeds = SEEDS[:1]
for exp in EXPERIMENTS[:2]:
    for model in preview_models:
        prof = profile_for(model, exp['profile'])
        for seed in preview_seeds:
            if exp['kind'] == 'train_eval':
                run_train_eval(exp['name'], model, seed, exp['train'], exp['val'], exp['target'], exp.get('source'), prof, dry_run=True)
            else:
                run_eval_only(exp['name'], model, seed, exp['csv'], exp['checkpoint_experiment'], prof, dry_run=True)


In [ ]:
#@title 9) Run benchmark matrix (resumable)
if not RUN_EXPERIMENTS:
    print('RUN_EXPERIMENTS is False. This is dry-run only. Set it True in cell 3 to start training/evaluation.')

for exp in EXPERIMENTS:
    for model in MODEL_LIST:
        prof = profile_for(model, exp['profile'])
        for seed in SEEDS:
            if exp['kind'] == 'train_eval':
                run_train_eval(exp['name'], model, seed, exp['train'], exp['val'], exp['target'], exp.get('source'), prof, dry_run=not RUN_EXPERIMENTS)
            else:
                run_eval_only(exp['name'], model, seed, exp['csv'], exp['checkpoint_experiment'], prof, dry_run=not RUN_EXPERIMENTS)


In [ ]:
#@title 10) Summarize completed runs
SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)
for exp in EXPERIMENTS:
    for model in MODEL_LIST:
        root_dir = BENCH_ROOT / exp['name'] / model
        if not root_dir.exists():
            continue
        for eval_subdir in ['eval_target', 'eval_source']:
            if not list(root_dir.glob(f'seed_*/{eval_subdir}/{model}_eval.json')):
                continue
            out_dir = SUMMARY_ROOT / exp['name'] / eval_subdir / model
            cmd = f'''python /content/DADS7202_PigPicture/summarize_seed_runs.py \
  --root-dir "{root_dir}" \
  --model {model} \
  --eval-subdir {eval_subdir} \
  --output-dir "{out_dir}"'''
            run_cmd(cmd, LOG_ROOT / exp['name'] / model / f'summary_{eval_subdir}.log', dry_run=False)


In [ ]:
#@title 11) Build combined result tables
import pandas as pd, json
rows = []
for csv_path in SUMMARY_ROOT.glob('*/*/*/*_seed_runs.csv'):
    parts = csv_path.relative_to(SUMMARY_ROOT).parts
    exp_name, eval_subdir, model = parts[0], parts[1], parts[2]
    df = pd.read_csv(csv_path)
    df.insert(0, 'experiment', exp_name)
    df.insert(1, 'eval_split', eval_subdir)
    df.insert(2, 'model', model)
    rows.append(df)

if rows:
    all_runs = pd.concat(rows, ignore_index=True)
    all_runs.to_csv(SUMMARY_ROOT / 'all_seed_runs_long.csv', index=False)
    metrics = ['accuracy','macro_precision','macro_recall','macro_f1','loss']
    summary = all_runs.groupby(['experiment','eval_split','model'])[metrics].agg(['count','mean','std','min','max'])
    summary.columns = ['_'.join(c) for c in summary.columns]
    summary = summary.reset_index()
    summary.to_csv(SUMMARY_ROOT / 'all_experiment_model_summary.csv', index=False)
    display(summary)
    print('saved:', SUMMARY_ROOT / 'all_seed_runs_long.csv')
    print('saved:', SUMMARY_ROOT / 'all_experiment_model_summary.csv')
else:
    print('No completed summary CSV files found yet.')


In [ ]:
#@title 12) Progress and missing jobs
import pandas as pd
progress = []
for exp in EXPERIMENTS:
    for model in MODEL_LIST:
        for seed in SEEDS:
            root = BENCH_ROOT / exp['name'] / model / f'seed_{seed}'
            if exp['kind'] == 'train_eval':
                train_done = model_checkpoint(root / 'train', model).exists()
                target_done = eval_json(root / 'eval_target', model).exists()
                source_done = eval_json(root / 'eval_source', model).exists()
            else:
                train_done = None
                target_done = None
                source_done = eval_json(root / 'eval_source', model).exists()
            progress.append(dict(experiment=exp['name'], model=model, seed=seed, train_done=train_done, target_done=target_done, source_done=source_done))
progress_df = pd.DataFrame(progress)
progress_df.to_csv(SUMMARY_ROOT / 'progress_table.csv', index=False)
display(progress_df.groupby(['experiment','model'])[['train_done','target_done','source_done']].sum(numeric_only=True).reset_index())
print('saved:', SUMMARY_ROOT / 'progress_table.csv')


In [ ]:
#@title 13) Optional cleanup note
print('No cleanup is performed by this notebook.')
print('Artifacts are under:', BENCH_ROOT)
print('Logs are under:', LOG_ROOT)
